In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skewnorm
import seaborn as sns
from efficient_apriori import apriori
import re
import numpy as np
from scipy import stats

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
datas = pd.read_csv('PSCompPars_2025.06.12_09.54.16.csv', comment='#', low_memory=False)
data = pd.DataFrame(data=datas)
origdata = data.copy(deep = True)
#These two were labeled incorrectly on the dataframe
data.replace({ 'st_spectype': {'m3 V':'M V'} }, inplace = True)
data.replace({ 'st_spectype': {'sdBV':'B V'} }, inplace = True)

In [3]:
#assigns random temperature for each planet
def rand_temp(row):
    try:
        mean = row['pl_eqt'] + ((row['pl_eqterr1']+row['pl_eqterr2'])/2)
        sd = row['pl_eqt'] + row['pl_eqterr1'] - mean
        a = (3*(mean-row['pl_eqt']))/sd
        return skewnorm.rvs(a, loc=mean, scale=row['pl_eqterr1'], size=1)[0]
    except:
        return row['pl_eqt']

#assigns random planet radius:  
def rand_prade(row):
    try:
        mean = row['pl_rade'] + ((row['pl_radeerr1']+row['pl_radeerr2'])/2)
        sd = row['pl_rade'] + row['pl_radeerr1'] - mean
        a = (3*(mean-row['pl_rade']))/sd
        return skewnorm.rvs(a, loc=mean, scale=row['pl_radeerr1'], size=1)[0]
    except:
        return row['pl_rade']

#assigns random planet semimajor axis: 
def rand_majoraxis(row):
    try:
        mean = row['pl_orbsmax'] + ((row['pl_orbsmaxerr1']+row['pl_orbsmaxerr2'])/2)
        sd = row['pl_orbsmax'] + row['pl_orbsmaxerr1'] - mean
        a = (3*(mean-row['pl_orbsmax']))/sd
        return skewnorm.rvs(a, loc=mean, scale=row['pl_orbsmaxerr1'], size=1)[0]
    except: 
        return row['pl_orbsmax']

#assigns random planet orbital period:
def rand_orbper(row):
    try:
        mean = row['pl_orbper'] + ((row['pl_orbpererr1']+row['pl_orbpererr2'])/2)
        sd = row['pl_orbper'] + row['pl_orbpererr1'] - mean
        a = (3*(mean-row['pl_orbper']))/sd
        return skewnorm.rvs(a, loc=mean, scale=row['pl_orbpererr1'], size=1)[0]
    except: 
        return row['pl_orbper']

#estimates the semimajor axis given a row of a data frame given stellar mass and orbital period: Kepler's 3rd law
#Only estimates the semimajor axis if the value is originally NaN
def estimate_orbsmax(row):
    if pd.isna(row['pl_orbsmax']):
        try: 
            return (((6.67e-11 * (row['st_mass'] * 1.989e30) * ((row['pl_orbper'] * 86400)**2)) / (4 * (np.pi**2)))**(1/3)) / 1.496e11
        except:
            return np.nan
    else:
        return row['pl_orbsmax']
        

#Sets the spectral type if it is already in the string for the spectype variable for each row.
#Otherwise, it is estimated based on equillibrium temperature.
def get_spectype(row):
    try: 
        return row['st_spectype'][0]
    except:
        if(row['st_teff']>=30000):
            return 'O'
        elif(row['st_teff']>=10000):
            return 'B'
        elif(row['st_teff']>=7500):
            return 'A'
        elif(row['st_teff']>=6000):
            return 'F'
        elif(row['st_teff']>=5200):
            return 'G'
        elif(row['st_teff']>=3700):
            return 'K'
        elif(row['st_teff']>=2400):
            return 'M'
        else:
            return row['st_spectype']

#Estimates the equillibrium temperature of a row given stellar luminosity, and semimajor axis
#Only estimates if there is a missing value for pl_eqt: Stefan-Boltzmann's law
def estimate_pl_eqt(row):
    if pd.isna(row['pl_eqt']):
        try:
            return (((10 ** row['st_lum']) * 3.827e26) / (16 * np.pi * 5.67e-8 * ((row['pl_orbsmax'] * 1.496e11) ** 2))) ** (1 / 4)
        except:
            return np.nan
    else:
        return row['pl_eqt']


#Grabs the last value of the string, usually the Luminosity Class of the star
#Otherwise, estimates whethoer or not the star is main sequence based on the luminosity of the star
def is_mainsequence(row):
    if(str(row['st_spectype'])[-1] == ('V') and str(row['st_spectype'])[-2:] != ('IV')):
        return 'True'
    else:
        try:
            if(str(row['spectype']).__contains__('M') and row['st_lum'] < -1.0969):
                return 'True'
            elif(str(row['spectype']).__contains__('K') and row['st_lum'] < -0.2218):
                return 'True'
            elif(str(row['spectype']).__contains__('G') and row['st_lum'] < 0.1761):
                return 'True'
            elif(str(row['spectype']).__contains__('F') and row['st_lum'] < 0.699):
                return 'True'
            elif(str(row['spectype']).__contains__('A') and row['st_lum'] < 1.3979):
                return 'True'
            elif(str(row['spectype']).__contains__('B') and row['st_lum'] < 4.477):
                return 'True'
            elif(str(row['spectype']).__contains__('O')):
                return 'True'
            else:
                return 'False'
        except:
            return 'False'   

#Gets the right hand and left hand side of the rule and puts it in an items list
pl_pattern = re.compile(r'(.+?)(\d)')
def parse_rule(rule_str):
    lhs, rhs = rule_str.split('->')
    lhs_items = [item.strip() for item in lhs.strip('{} ').split(',')]
    rhs = rhs.split("{")[1].split("}")[0]
    rhs_items = [item.strip() for item in rhs.strip('{} ').split(',')]
    return lhs_items, rhs_items

#Puts items in a system into a dictionary so that they are easy to compare, get counts for classifications
def get_plnum_dict(items):
    orbital_dict = {}
    for item in items:
        match = pl_pattern.match(item)
        if match:
            base_type, number = match.groups()
            number = int(number)
            orbital_dict.setdefault(base_type, set()).add(number)
    return orbital_dict

In [ ]:
data = origdata.copy(deep= True)
#applying all of the non-random variables
data['pl_orbsmax'] = data.apply(estimate_orbsmax, axis=1)
data['pl_eqt'] = data.apply(estimate_pl_eqt, axis = 1)
data['spectype'] = data.apply(get_spectype, axis = 1)
data['isMS'] = data.apply(is_mainsequence, axis = 1)
#Remove systems with more that one host star
star = data.loc[data['sy_snum'] < 2]
# Excluded planet systems that did not have any data on the variables of interest
star = star.loc[star['pl_orbper'].isna() == False]
star = star.loc[star['pl_rade'].isna() == False]
star = star.loc[star['st_teff'].isna() == False]
star = star.loc[star['pl_eqt'].isna() == False]
star = star.loc[star['spectype'].isna() == False]
star['pl_rad_bin'] = pd.cut(star['pl_rade'], bins = [0, 1.7, 3.9, 9.4, 5000], labels = ['Terrestrial', 'Mini-Neptune', 'Sub-Saturn', 'Gas Giant'])
star['pl_orb_bin'] = pd.cut(star['pl_orbper'], bins = [0, 10, 100, 1000, 10000000000], labels = ['Short Orbital', 'Medium-Short Orbital', 'Medium-Long Orbital', 'Long Orbital'])
star['pl_eqt_bin'] = pd.cut(star['pl_eqt'], bins = [0, 200, 400, 1000, 10000000], labels = ['Cold', 'Temperate', 'Warm', 'Hot'])
star['pl_class'] = star[['pl_eqt_bin','pl_rad_bin', 'pl_orb_bin']].astype('str').agg(" ".join, axis=1)
star['stars_class'] = star[['spectype', 'pl_class']].agg(" ".join, axis=1)
#Creating a dataframe with only main sequence stars and their proper classificaiton
onlyMS = star.loc[star['isMS'] == 'True']

#originally 55 only MS and pl class
'''test = {'type': onlyMS['pl_class'],'host':onlyMS['hostname']}'''

#originally 92 only MS and pl+star class
test = {'type': onlyMS['stars_class'],'host':onlyMS['hostname']}

#originally 70 all stars and pl class
'''test = {'type': star['pl_class'],'host':star['hostname']}'''

#Originally 103 all stars and pl+st class
'''test = {'type': star['stars_class'],'host':star['hostname']}'''

testing = pd.DataFrame(data=test)
repeats = testing.groupby('host')

#Put into tuples by host star for the package
modified_tuples = []
for host, group in repeats:
    seen = {}
    new_types = []
    for t in group['type']:
        if t not in seen:
            seen[t] = 1
            new_types.append(f"{t}{seen[t]}")
        else:
            seen[t] += 1
            new_types.append(f"{t}{seen[t]}")
    modified_tuples.append(tuple(new_types))

origitemset, origrules = apriori(modified_tuples, min_support= 2/len(modified_tuples), min_confidence=.5)
filtered_rules = []
#goes through every rule created using the cheat method and removes rules whose left side does not have sequential orders of planets: ex. 3 without 2 or 1
origrules = sorted(origrules, key=lambda rule: rule.confidence)
for rule in origrules:
    rule_str = str(rule)
    lhs_items, rhs_items = parse_rule(rule_str)
    lhs_plnum = get_plnum_dict(lhs_items)
    rhs_plnum = get_plnum_dict(rhs_items)
    all_items = lhs_items + rhs_items
    all_plnum = get_plnum_dict(all_items)
    rule_valid = True
    for base_type, plnums in lhs_plnum.items():
        for num in plnums:
            if num > 1:
                required = set(range(1, num))
                if not required.issubset(plnums):
                    rule_valid = False
                    break
        if not rule_valid:
            break
    for base_type, plnum in rhs_plnum.items():
        if not rule_valid:
            break
        for items in plnum:
            if items > 1:
                required = set(range(1, items))
                if not required.issubset(all_plnum[base_type]):
                    rule_valid = False
                    break
    if rule_valid:
        filtered_rules.append(rule)
filtered_rules = sorted(filtered_rules, key=lambda rule: rule.lift)

listorig = []
for rule in filtered_rules:
    listorig.append(rule)
print('There are ' + str(listorig.__len__())+ ' rules.')
for rule in filtered_rules:
    print(rule)

ValueError: `min_support` must be a number between 0 and 1.

In [ ]:
#shows planet matches between the rules and actual systems. Gives the planets in the system but not the actual name. That is below.
rulepredictions = {}
for rule in filtered_rules:
    lhs_items, rhs_items = parse_rule(str(rule))
    for tuple in modified_tuples:
        if all(lhs in tuple for lhs in lhs_items):
            if all(rhs not in tuple for rhs in rhs_items):
                if str(rule) in rulepredictions:
                    rulepredictions[str(rule)].append(tuple)
                else:
                    rulepredictions[str(rule)] = [tuple]
                
rulepredictions

{'{G Hot Terrestrial Short Orbital1, G Hot Terrestrial Short Orbital2, G Warm Mini-Neptune Short Orbital1} -> {G Warm Mini-Neptune Medium-Short Orbital1} (conf: 0.500, supp: 0.001, lift: 3.428, conv: 1.708)': [('G Warm Mini-Neptune Short Orbital1',
   'G Hot Terrestrial Short Orbital1',
   'G Hot Terrestrial Short Orbital2'),
  ('G Hot Terrestrial Short Orbital1',
   'G Hot Terrestrial Short Orbital2',
   'G Warm Mini-Neptune Short Orbital1')],
 '{G Warm Mini-Neptune Short Orbital1, G Warm Terrestrial Short Orbital1} -> {G Warm Mini-Neptune Medium-Short Orbital1} (conf: 0.500, supp: 0.001, lift: 3.428, conv: 1.708)': [('G Hot Terrestrial Short Orbital1',
   'G Warm Mini-Neptune Short Orbital1',
   'G Warm Terrestrial Short Orbital1'),
  ('G Warm Terrestrial Short Orbital1',
   'G Warm Mini-Neptune Short Orbital1',
   'G Hot Terrestrial Short Orbital1'),
  ('G Warm Terrestrial Short Orbital1', 'G Warm Mini-Neptune Short Orbital1')],
 '{G Hot Mini-Neptune Short Orbital1, G Warm Sub-Satur

In [ ]:
#storing the names of the systems in a database for later
systems_list = []
systems = repeats = testing.groupby('host')
for host, group in repeats:
    seen = {}
    new_types = []
    for t in group['type']:
        if t not in seen:
            seen[t] = 1
            new_types.append(f"{t}{seen[t]}")
        else:
            seen[t] += 1
            new_types.append(f"{t}{seen[t]}")
    systems_list.append([host, new_types])
systems_list

[['14 Her',
  ['K Cold Gas Giant Long Orbital1', 'K Cold Gas Giant Long Orbital2']],
 ['47 UMa',
  ['G Temperate Gas Giant Long Orbital1',
   'G Cold Gas Giant Long Orbital1',
   'G Cold Gas Giant Long Orbital2']],
 ['51 Peg', ['G Hot Gas Giant Short Orbital1']],
 ['61 Vir',
  ['G Hot Mini-Neptune Short Orbital1',
   'G Warm Sub-Saturn Medium-Short Orbital1',
   'G Temperate Sub-Saturn Medium-Long Orbital1']],
 ['70 Vir', ['G Warm Gas Giant Medium-Long Orbital1']],
 ['AF Lep', ['F Hot Gas Giant Long Orbital1']],
 ['BD+14 4559', ['K Temperate Gas Giant Medium-Long Orbital1']],
 ['BD+20 594', ['G Warm Mini-Neptune Medium-Short Orbital1']],
 ['BD+45 564', ['K Temperate Gas Giant Medium-Long Orbital1']],
 ['BD+55 362', ['K Temperate Gas Giant Medium-Long Orbital1']],
 ['BD+63 1405', ['K Cold Gas Giant Long Orbital1']],
 ['BD-06 1339',
  ['K Warm Mini-Neptune Short Orbital1',
   'K Temperate Sub-Saturn Medium-Long Orbital1']],
 ['BD-08 2823',
  ['K Warm Mini-Neptune Short Orbital1',
   'K T

In [ ]:
#Gives the actual names of the systems.
systempredictions = {}
for rule in filtered_rules:
    lhs_items, rhs_items = parse_rule(str(rule))
    for lister in systems_list:
        tuple = lister[1]
        if all(lhs in tuple for lhs in lhs_items):
            if all(rhs not in tuple for rhs in rhs_items):
                if (str(lhs_items)+' -> '+str(rhs_items)) in systempredictions:
                    systempredictions[str(lhs_items)+' -> '+str(rhs_items)].append(lister[0])
                else:
                    systempredictions[str(lhs_items)+' -> '+str(rhs_items)] = [lister[0]]
systempredictions

{"['G Hot Terrestrial Short Orbital1', 'G Hot Terrestrial Short Orbital2', 'G Warm Mini-Neptune Short Orbital1'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['Kepler-1073',
  'Kepler-1530'],
 "['G Warm Mini-Neptune Short Orbital1', 'G Warm Terrestrial Short Orbital1'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['Kepler-226',
  'Kepler-255',
  'Kepler-783'],
 "['G Hot Mini-Neptune Short Orbital1', 'G Warm Sub-Saturn Medium-Short Orbital1'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['61 Vir',
  'Kepler-18',
  'Kepler-82'],
 "['G Temperate Mini-Neptune Medium-Long Orbital1', 'G Warm Mini-Neptune Short Orbital1'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['Kepler-920'],
 "['G Temperate Sub-Saturn Medium-Long Orbital1', 'G Warm Sub-Saturn Medium-Long Orbital1'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['HD 34445'],
 "['G Hot Mini-Neptune Short Orbital1', 'G Hot Mini-Neptune Short Orbital2'] -> ['G Warm Mini-Neptune Medium-Short Orbital1']": ['Keple

In [ ]:
#outputs the number of planets in agreement between a givin rule and a system. For example, 2/2 -> 2 out of 2 planets in the system are found in the rule, so we predict the following planets.
rulespredictions = {}
for rule in filtered_rules:
    lhs_items, rhs_items = parse_rule(str(rule))
    for lister in systems_list:
        tuple = lister[1]
        if all(lhs in tuple for lhs in lhs_items):
            if all(rhs not in tuple for rhs in rhs_items):
                if (lister[0]) in rulespredictions:
                    rulespredictions[lister[0]].append(str(lhs_items.__len__())+'/'+ str(lister[1].__len__()) + ' '+str(rhs_items))
                else:
                    rulespredictions[lister[0]] = [str(lhs_items.__len__())+'/'+ str(lister[1].__len__()) + ' '+str(rhs_items)]
for entry in rulespredictions:
    if(rulespredictions[entry].__len__()>1):
        print(str(entry) +' ' +str(rulespredictions[entry]))

Kepler-920 ["2/2 ['G Warm Mini-Neptune Medium-Short Orbital1']", "2/2 ['G Warm Mini-Neptune Medium-Short Orbital1', 'G Warm Mini-Neptune Medium-Short Orbital2']"]
GJ 9827 ["3/3 ['K Warm Mini-Neptune Medium-Short Orbital1']", "3/3 ['K Warm Terrestrial Short Orbital2']", "3/3 ['K Warm Mini-Neptune Short Orbital2']"]
Kepler-1591 ["1/1 ['F Warm Mini-Neptune Medium-Short Orbital1']", "1/1 ['F Hot Terrestrial Short Orbital1']"]
Kepler-1969 ["1/1 ['F Warm Mini-Neptune Medium-Short Orbital1']", "1/1 ['F Hot Terrestrial Short Orbital1']"]
Kepler-431 ["1/3 ['F Warm Mini-Neptune Medium-Short Orbital1']", "2/3 ['F Warm Mini-Neptune Medium-Short Orbital1']", "2/3 ['F Warm Mini-Neptune Medium-Short Orbital1', 'F Warm Mini-Neptune Medium-Short Orbital2']"]
Kepler-1513 ["1/2 ['G Hot Gas Giant Short Orbital1']", "1/2 ['G Temperate Gas Giant Medium-Long Orbital1']"]
Kepler-1165 ["1/2 ['F Hot Terrestrial Short Orbital1']", "2/2 ['F Hot Terrestrial Short Orbital1']", "2/2 ['F Warm Mini-Neptune Medium-Shor